# Pneumonia Detection - GPU Training (Google Colab)

This notebook trains the full transfer-learning suite on the **complete** RSNA dataset using Colab's free GPU, then compares models, fine-tunes the winner, and saves the best model. It reuses the tested `src/` package from the project repository.

**Before you start:** Runtime -> Change runtime type -> Hardware accelerator -> **GPU**.

## 1. Confirm GPU is available

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus, 'No GPU! Set Runtime -> Change runtime type -> GPU.'

## 2. Get the code
Clone your project repository (it contains the tested `src/` package and the app). Replace the URL if your repo differs.

In [ ]:
!git clone https://github.com/satyasarthak/pneumonia-detection.git
%cd pneumonia-detection
!pip -q install pydicom

## 3. Get the data
The DICOM archives are NOT in the repo (too large). Choose **ONE** of the two options below.

### Option A - Google Drive (simplest, recommended)
1. In your Google Drive, create a folder named `pneumonia_data`.
2. Upload these two files into it (from your local project folder):
   - `stage_2_detailed_class_info.csv`
   - `stage_2_train_images.zip`
3. Run the cell below and approve the Drive mount popup.

The zip is ~3.7 GB, so the Drive upload can take a while - do it before class/overnight. Once uploaded it stays there for reuse.

In [ ]:
# --- Option A: Google Drive ---
from google.colab import drive
import shutil, os
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/pneumonia_data'  # <- the folder you created
assert os.path.isdir(DRIVE), f'Folder not found: {DRIVE}. Create it and upload the files.'
shutil.copy(os.path.join(DRIVE, 'stage_2_detailed_class_info.csv'), '.')
shutil.copy(os.path.join(DRIVE, 'stage_2_train_images.zip'), '.')
print('CSV present:', os.path.exists('stage_2_detailed_class_info.csv'))
print('ZIP present:', os.path.exists('stage_2_train_images.zip'))

### Option B - Kaggle API (skip if you used Option A)
Downloads the data straight into Colab (fast, no manual upload). You need a Kaggle account and to have accepted the competition rules.
1. Kaggle -> Account -> Create New API Token -> downloads `kaggle.json`.
2. Run the cell, upload `kaggle.json` when prompted.

In [ ]:
# --- Option B: Kaggle (only run if you did NOT use Option A) ---
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip -q install kaggle
# !kaggle competitions download -c rsna-pneumonia-detection-challenge -f stage_2_detailed_class_info.csv
# !kaggle competitions download -c rsna-pneumonia-detection-challenge -f stage_2_train_images.zip
# !unzip -o -q '*.zip.zip' 2>/dev/null; print('done')

## 4. Prepare the dataset
Load labels, de-duplicate patients, restrict to the three classes, resolve images, and split (stratified). Uses the FULL dataset.

In [ ]:
import pandas as pd
from src.labels import load_labels, deduplicate_patients, restrict_labels
from src.images import resolve_images
from src.split import stratified_split

clean = restrict_labels(deduplicate_patients(load_labels('stage_2_detailed_class_info.csv', verbose=False)))
present, missing = resolve_images(clean, 'stage_2_train_images.zip')
print('Patients:', len(present), '| missing images:', len(missing))

train_df, val_df, test_df = stratified_split(present)
print('Train/Val/Test:', len(train_df), len(val_df), len(test_df))

## 5. Train the transfer-learning suite (full data, GPU)
We train two pretrained backbones with the standard head, plus a DEEPER custom-head architecture (added layers), all at 224x224 with augmentation and class weights. On a GPU each is fast.

In [ ]:
from src.train import train_transfer, train_transfer_finetune
from src.generator import XrayBatchGenerator
from src.evaluate import evaluate_model, compare_models, select_best, commentary
from src.models import build_transfer_model_deep
from src.train import make_generators, compute_class_weights, _default_callbacks

SIZE = (224, 224)
BATCH = 32
EPOCHS = 15
results, models = {}, {}

In [ ]:
# --- MobileNetV2 (standard head, frozen base) ---
m, _ = train_transfer('MobileNetV2', train_df, val_df, 'stage_2_train_images.zip',
                      image_size=SIZE, batch_size=BATCH, epochs=EPOCHS)
models['MobileNetV2'] = m
tg = XrayBatchGenerator(test_df, 'stage_2_train_images.zip', batch_size=BATCH,
                        target_channels=3, image_size=SIZE, shuffle=False)
results['MobileNetV2'] = evaluate_model(m, tg, 'MobileNetV2')
print(commentary(results['MobileNetV2']))

In [ ]:
# --- ResNet50 (standard head, frozen base) ---
m, _ = train_transfer('ResNet50', train_df, val_df, 'stage_2_train_images.zip',
                      image_size=SIZE, batch_size=BATCH, epochs=EPOCHS)
models['ResNet50'] = m
tg = XrayBatchGenerator(test_df, 'stage_2_train_images.zip', batch_size=BATCH,
                        target_channels=3, image_size=SIZE, shuffle=False)
results['ResNet50'] = evaluate_model(m, tg, 'ResNet50')
print(commentary(results['ResNet50']))

In [ ]:
# --- MobileNetV2 with a DEEPER custom head (new architecture, added layers) ---
deep = build_transfer_model_deep('MobileNetV2', input_shape=(*SIZE, 3))
tr_gen, va_gen = make_generators(train_df, val_df, 'stage_2_train_images.zip',
                                 target_channels=3, image_size=SIZE, batch_size=BATCH)
deep.fit(tr_gen, validation_data=va_gen, epochs=EPOCHS,
         class_weight=compute_class_weights(train_df),
         callbacks=_default_callbacks(), verbose=2)
models['MobileNetV2-Deep'] = deep
tg = XrayBatchGenerator(test_df, 'stage_2_train_images.zip', batch_size=BATCH,
                        target_channels=3, image_size=SIZE, shuffle=False)
results['MobileNetV2-Deep'] = evaluate_model(deep, tg, 'MobileNetV2-Deep')
print(commentary(results['MobileNetV2-Deep']))

In [ ]:
# --- Fine-tuned MobileNetV2 (two-phase: head, then unfreeze top of backbone) ---
ft, hist = train_transfer_finetune('MobileNetV2', train_df, val_df,
                                   'stage_2_train_images.zip', image_size=SIZE,
                                   batch_size=BATCH, head_epochs=8, finetune_epochs=12)
models['MobileNetV2-FineTuned'] = ft
tg = XrayBatchGenerator(test_df, 'stage_2_train_images.zip', batch_size=BATCH,
                        target_channels=3, image_size=SIZE, shuffle=False)
results['MobileNetV2-FineTuned'] = evaluate_model(ft, tg, 'MobileNetV2-FineTuned')
print(commentary(results['MobileNetV2-FineTuned']))

## 6. Compare all models and select the best

In [ ]:
table = compare_models(results).sort_values('macro_f1', ascending=False)
display(table.round(4))
best_name, rationale = select_best(table, metric='macro_f1')
print('BEST:', best_name)
print(rationale)

In [ ]:
# Per-class detail for the best model.
import pandas as pd
pd.DataFrame(results[best_name]['per_class']).T.round(4)

## 7. Serialize the best model, reload, and run inference

In [ ]:
from src.registry import save_model, load_model
import numpy as np

save_model(models[best_name], 'models/best_model.keras')
# Record the geometry so the Streamlit app auto-configures for this model.
with open('models/best_model_geometry.txt', 'w') as fh:
    fh.write(f'3,{SIZE[0]},{SIZE[1]}')
reloaded = load_model('models/best_model.keras')

tg = XrayBatchGenerator(test_df.head(6), 'stage_2_train_images.zip', batch_size=6,
                        target_channels=3, image_size=SIZE, shuffle=False)
x, y = tg[0]
before = models[best_name].predict(x, verbose=0)
after = reloaded.predict(x, verbose=0)
print('Round-trip identical:', np.allclose(before, after, atol=1e-6))

from src.constants import INDEX_TO_CLASS
for i in range(x.shape[0]):
    p = after[i]; idx = int(p.argmax())
    print(f'true={INDEX_TO_CLASS[int(y[i].argmax())]:<28} pred={INDEX_TO_CLASS[idx]:<28} p={p[idx]:.2f}')

## 8. Save results back to Drive
Download `models/best_model.keras` and commit it to your repo so the Streamlit app serves the strong model. Also save the comparison table for your report.

In [ ]:
import shutil, os
os.makedirs('outputs', exist_ok=True)
table.to_csv('outputs/model_comparison_final.csv', index=False)

# If Drive was mounted (Option A), copy artifacts there for easy download.
if 'DRIVE' in globals() and os.path.isdir(DRIVE):
    shutil.copy('models/best_model.keras', DRIVE)
    shutil.copy('models/best_model_geometry.txt', DRIVE)
    shutil.copy('outputs/model_comparison_final.csv', DRIVE)
    print('Saved best_model.keras, geometry, and comparison to your Drive folder.')
else:
    from google.colab import files
    files.download('models/best_model.keras')
    files.download('models/best_model_geometry.txt')
    files.download('outputs/model_comparison_final.csv')
print('Next: replace models/best_model.keras + best_model_geometry.txt in your repo and push.')

---
### After training
1. Download `best_model.keras` (from Drive or the file browser).
2. In your local repo, replace `models/best_model.keras` with this file and update the app geometry if needed (it is RGB 224x224).
3. Commit and push, then redeploy the Codespace so the app serves the strong model.
4. Copy the comparison table and per-class metrics into your final report.